# Turb-DETR Week 2B: Training (Checkpoint-Safe)

**GPU required. Run AFTER Week 2A.**

**Designed for 4.5 hour GPU limit:**
- Phase 1: Data rebuild (~40 min)
- Phase 2: Turb-DETR training with checkpoints every 5 epochs
- Phase 3: Evaluation (can run in separate session if time runs out)

**Checkpoint strategy:** Weights saved to Drive every 5 epochs. If session dies, resume from last checkpoint in a new session.


In [1]:
# ============================================================
# CELL 1: Setup + restore patched Ultralytics
# ============================================================
import os, time, shutil
t0 = time.time()
import torch
if not torch.cuda.is_available(): raise RuntimeError("NO GPU")
print(f"GPU: {torch.cuda.get_device_name(0)}")

!pip install -q ultralytics==8.3.40

from google.colab import drive
drive.mount('/content/drive')

import ultralytics
from pathlib import Path

# Restore patched file from Week 2A
PREP = Path("/content/drive/MyDrive/turb_detr_results/week2_prep")

# Check which file was patched (block.py or head.py)
patched_file = None
for fname in ["head_patched.py", "block_patched.py", "head.py", "block.py"]:
    if (PREP / fname).exists():
        patched_file = fname
        break

if patched_file is None:
    print("ERROR: No patched file found in", PREP)
    print("Available:", list(PREP.glob("*.py")))
    raise FileNotFoundError("Run Week 2A first!")

# Determine destination
ul_path = Path(ultralytics.__file__).parent / "nn" / "modules"
if "head" in patched_file:
    dst = ul_path / "head.py"
else:
    dst = ul_path / "block.py"

shutil.copy2(PREP / patched_file, dst)
print(f"Restored: {patched_file} -> {dst.name}")

# Verify SimAM
source = dst.read_text()
assert "SimAM" in source or "simam" in source, "SimAM not found in patched file!"
print("SimAM verified")

DATASET = "/content/drive/MyDrive/underwater_datasets/trash_ICRA19/dataset"
if not os.path.exists(DATASET):
    base = "/content/drive/MyDrive/underwater_datasets"
    for n in os.listdir(base):
        if "icra" in n.lower():
            for sub in ["dataset", ""]:
                cand = os.path.join(base, n, sub) if sub else os.path.join(base, n)
                if os.path.exists(os.path.join(cand, "train")):
                    DATASET = cand; break

os.makedirs("/content/turb-detr", exist_ok=True)
os.chdir("/content/turb-detr")
print(f"Dataset: {DATASET}")
print(f"Setup: {time.time()-t0:.1f}s")


Mounted at /content/drive
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Restored: head.py -> head.py
SimAM verified
Dataset: /content/drive/MyDrive/underwater_datasets/trash_ICRA19/dataset
Setup: 42.1s


In [2]:
# ============================================================
# CELL 2: Rebuild all data (augmentation + combined set)
# This repeats Week 2A work because Colab resets between sessions
# ============================================================
import os, subprocess, shutil, cv2, numpy as np, time
from pathlib import Path
from tqdm import tqdm
t0 = time.time()

def jaffe(image, c=0.15, depth=3.0, seed=None):
    rng = np.random.RandomState(seed)
    img = image.astype(np.float32) / 255.0
    att = np.exp(-c * depth)
    bs = np.zeros_like(img)
    bs[:,:,0] = 0.10*(1-att); bs[:,:,1] = 0.30*(1-att); bs[:,:,2] = 0.05*(1-att)
    t = img * att + bs
    n = rng.normal(0, 0.02*c*10, img.shape).astype(np.float32)
    return (np.clip(t+n, 0, 1)*255).astype(np.uint8)

SRC = Path(DATASET); OUT = Path("data/processed"); LOCAL = Path("/content/local_labels")
LEVELS = {"light": 0.05, "medium": 0.15, "heavy": 0.30}

# Step 1: Restructure + filter ROV
print("Step 1: Restructure + filter ROV...")
for split in ["train", "val", "test"]:
    sd = SRC/split
    (OUT/split/"images").mkdir(parents=True, exist_ok=True)
    (OUT/split/"labels").mkdir(parents=True, exist_ok=True)
    for f in sd.glob("*.jpg"):
        d = OUT/split/"images"/f.name
        if not d.exists(): os.symlink(f.resolve(), d)
    ll = LOCAL/split; ll.mkdir(parents=True, exist_ok=True)
    subprocess.run(f"cp {sd}/*.txt {ll}/ 2>/dev/null", shell=True)
    for txt in ll.glob("*.txt"):
        lines = [l.strip() for l in open(txt) if len(l.strip().split())==5 and l.strip().split()[0]!="2"]
        with open(OUT/split/"labels"/txt.name, "w") as f:
            f.write("\n".join(lines)+"\n" if lines else "")
    print(f"  {split} done")

bad = OUT/"train"/"images"/"obj1658_frame0002383 (1).jpg"
if bad.exists() or bad.is_symlink(): os.unlink(bad)

# Step 2: Augment training images
train_imgs = sorted((OUT/"train"/"images").glob("*.jpg"))
print(f"\nStep 2: Augment {len(train_imgs)} train imgs x3...")
for ln, cv_val in LEVELS.items():
    oi = Path(f"data/augmented_train/{ln}/images"); oi.mkdir(parents=True, exist_ok=True)
    ol = Path(f"data/augmented_train/{ln}/labels"); ol.mkdir(parents=True, exist_ok=True)
    for i, ip in enumerate(tqdm(train_imgs, desc=f"train-{ln}")):
        img = cv2.imread(str(ip))
        if img is None: continue
        cv2.imwrite(str(oi/f"{ln}_{ip.name}"), jaffe(img, c=cv_val, seed=42+i))
        ls = OUT/"train"/"labels"/f"{ip.stem}.txt"
        if ls.exists(): shutil.copy2(ls, ol/f"{ln}_{ip.stem}.txt")

# Step 3: Augment test images
test_imgs = sorted((OUT/"test"/"images").glob("*.jpg"))
print(f"\nStep 3: Augment {len(test_imgs)} test imgs x3...")
for ln, cv_val in LEVELS.items():
    oi = Path(f"data/augmented_test/{ln}/images"); oi.mkdir(parents=True, exist_ok=True)
    ol = Path(f"data/augmented_test/{ln}/labels"); ol.mkdir(parents=True, exist_ok=True)
    for i, ip in enumerate(tqdm(test_imgs, desc=f"test-{ln}")):
        img = cv2.imread(str(ip))
        if img is None: continue
        cv2.imwrite(str(oi/ip.name), jaffe(img, c=cv_val, seed=42+i))
        ls = OUT/"test"/"labels"/f"{ip.stem}.txt"
        if ls.exists(): shutil.copy2(ls, ol/ls.name)

# Step 4: Combined training set
print("\nStep 4: Combined set...")
CI = Path("data/combined_train/images"); CI.mkdir(parents=True, exist_ok=True)
CL = Path("data/combined_train/labels"); CL.mkdir(parents=True, exist_ok=True)

for f in (OUT/"train"/"images").glob("*.jpg"):
    d = CI/f.name
    if not d.exists(): os.symlink(f.resolve(), d)
for f in (OUT/"train"/"labels").glob("*.txt"):
    d = CL/f.name
    if not d.exists(): os.symlink(f.resolve(), d)
for ln in LEVELS:
    for f in Path(f"data/augmented_train/{ln}/images").glob("*.jpg"):
        d = CI/f.name
        if not d.exists(): os.symlink(f.resolve(), d)
    for f in Path(f"data/augmented_train/{ln}/labels").glob("*.txt"):
        d = CL/f.name
        if not d.exists(): os.symlink(f.resolve(), d)

total = len(list(CI.glob("*.jpg")))
print(f"\nALL DATA READY ({time.time()-t0:.1f}s)")
print(f"  Combined training: {total} images")


Step 1: Restructure + filter ROV...
  train done
  val done
  test done

Step 2: Augment 5723 train imgs x3...


train-heavy: 100%|██████████| 5723/5723 [03:06<00:00, 30.67it/s]



Step 3: Augment 1144 test imgs x3...


test-heavy: 100%|██████████| 1144/1144 [00:37<00:00, 30.14it/s]



Step 4: Combined set...

ALL DATA READY (5950.6s)
  Combined training: 22892 images


In [3]:
# RUN THIS ONCE — saves augmented data to Drive (~5 min)
import subprocess, time
from pathlib import Path

t0 = time.time()
DRIVE_DATA = Path("/content/drive/MyDrive/turb_detr_results/augmented_data")
DRIVE_DATA.mkdir(parents=True, exist_ok=True)

# Save augmented training images as tar (much faster than individual files)
!tar cf - data/augmented_train/ | pigz > {DRIVE_DATA}/augmented_train.tar.gz
print(f"  augmented_train saved ({time.time()-t0:.0f}s)")

!tar cf - data/augmented_test/ | pigz > {DRIVE_DATA}/augmented_test.tar.gz
print(f"  augmented_test saved ({time.time()-t0:.0f}s)")

!tar cf - data/combined_train/ | pigz > {DRIVE_DATA}/combined_train.tar.gz
print(f"  combined_train saved ({time.time()-t0:.0f}s)")

# Also save the processed labels (filtered, no ROV)
!tar cf - data/processed/*/labels/ | pigz > {DRIVE_DATA}/processed_labels.tar.gz
print(f"  processed labels saved ({time.time()-t0:.0f}s)")

print(f"\nAll saved to {DRIVE_DATA}")
print("Total size:")
!du -sh {DRIVE_DATA}/*

  augmented_train saved (48s)
  augmented_test saved (58s)
  combined_train saved (58s)
  processed labels saved (59s)

All saved to /content/drive/MyDrive/turb_detr_results/augmented_data
Total size:
243M	/content/drive/MyDrive/turb_detr_results/augmented_data/augmented_test.tar.gz
1.2G	/content/drive/MyDrive/turb_detr_results/augmented_data/augmented_train.tar.gz
824K	/content/drive/MyDrive/turb_detr_results/augmented_data/combined_train.tar.gz
254K	/content/drive/MyDrive/turb_detr_results/augmented_data/processed_labels.tar.gz


In [ ]:
# FAST restore (~3 min instead of 40 min)
import os, subprocess, time
from pathlib import Path

t0 = time.time()
DRIVE_DATA = Path("/content/drive/MyDrive/turb_detr_results/augmented_data")

if DRIVE_DATA.exists() and (DRIVE_DATA / "augmented_train.tar.gz").exists():
    print("Restoring augmented data from Drive (fast)...")
    os.makedirs("data", exist_ok=True)
    
    !cd /content/turb-detr && pigz -dc {DRIVE_DATA}/augmented_train.tar.gz | tar xf -
    !cd /content/turb-detr && pigz -dc {DRIVE_DATA}/augmented_test.tar.gz | tar xf -
    !cd /content/turb-detr && pigz -dc {DRIVE_DATA}/combined_train.tar.gz | tar xf -
    !cd /content/turb-detr && pigz -dc {DRIVE_DATA}/processed_labels.tar.gz | tar xf -
    
    # Still need to symlink original images from Drive
    SRC = Path(DATASET)
    OUT = Path("data/processed")
    for split in ["train", "val", "test"]:
        (OUT/split/"images").mkdir(parents=True, exist_ok=True)
        for f in (SRC/split).glob("*.jpg"):
            d = OUT/split/"images"/f.name
            if not d.exists(): os.symlink(f.resolve(), d)
        print(f"  {split} images linked")
    
    bad = OUT/"train"/"images"/"obj1658_frame0002383 (1).jpg"
    if bad.exists() or bad.is_symlink(): os.unlink(bad)
    
    total = len(list(Path("data/combined_train/images").glob("*")))
    print(f"\nRestored in {time.time()-t0:.0f}s | Combined: {total} images")
    
else:
    print("No saved augmentation found. Generating from scratch...")
    # ... (keep the original augmentation code as fallback)

In [4]:
# ============================================================
# CELL 3: YAML configs
# ============================================================
import yaml, os
from pathlib import Path
os.makedirs("configs/datasets", exist_ok=True)

with open("configs/datasets/trash_icra19_clean.yaml","w") as f:
    yaml.dump({"path":str(Path("data/processed").resolve()),"train":"train/images","val":"val/images","test":"test/images","names":{0:"plastic",1:"bio"}},f)

with open("configs/datasets/trash_icra19_augmented.yaml","w") as f:
    yaml.dump({"path":str(Path("data").resolve()),"train":"combined_train/images","val":"processed/val/images","test":"processed/test/images","names":{0:"plastic",1:"bio"}},f)

for lv in ["light","medium","heavy"]:
    with open(f"configs/datasets/trash_icra19_turbid_{lv}.yaml","w") as f:
        yaml.dump({"path":str(Path(f"data/augmented_test/{lv}").resolve()),"train":"images","val":"images","test":"images","names":{0:"plastic",1:"bio"}},f)

print("All configs ready")


All configs ready


In [5]:
import random, numpy as np, torch, os
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
print(f"Seeds = {SEED}")


Seeds = 42


In [6]:
# ============================================================
# CELL 5: Check if we can resume from a previous checkpoint
# ============================================================
from pathlib import Path
import shutil

DRIVE_CKPT = Path("/content/drive/MyDrive/turb_detr_results/week2_checkpoints")
DRIVE_CKPT.mkdir(parents=True, exist_ok=True)

# Check if there's a checkpoint to resume from
resume_path = None
last_ckpt = DRIVE_CKPT / "turbdetr_augmented_last.pt"

if last_ckpt.exists():
    # Copy checkpoint to local for faster loading
    local_ckpt = Path("results/turbdetr_augmented/weights/last.pt")
    local_ckpt.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(last_ckpt, local_ckpt)
    resume_path = str(local_ckpt)
    print(f"FOUND CHECKPOINT: {last_ckpt}")
    print(f"  Will RESUME training from this checkpoint")
else:
    print("No checkpoint found. Training from scratch.")
    print(f"  Checkpoints will be saved to: {DRIVE_CKPT}")


No checkpoint found. Training from scratch.
  Checkpoints will be saved to: /content/drive/MyDrive/turb_detr_results/week2_checkpoints


## Training: Turb-DETR on Augmented Data

**20 epochs with checkpoints saved to Drive every 5 epochs.**

If session dies mid-training:
1. Open a NEW Colab session
2. Run this same notebook from Cell 1
3. Cell 5 will detect the checkpoint
4. Cell 6 will RESUME from where it stopped


In [ ]:
# ============================================================
# CELL 6: Train Turb-DETR with checkpoint saving
# ============================================================
import time, shutil
from pathlib import Path
from ultralytics import RTDETR

t0 = time.time()

DRIVE_CKPT = Path("/content/drive/MyDrive/turb_detr_results/week2_checkpoints")

if resume_path:
    print(f"RESUMING from checkpoint: {resume_path}")
    model = RTDETR(resume_path)
    model.train(
        resume=True,
        project="results",
        name="turbdetr_augmented",
        device=0,
    )
else:
    print("Training from scratch...")
    model = RTDETR("rtdetr-l.pt")
    model.train(
        data="configs/datasets/trash_icra19_augmented.yaml",
        epochs=20,
        imgsz=640,
        batch=16,
        seed=42,
        project="results",
        name="turbdetr_augmented",
        device=0,
        workers=2,
        patience=10,
        save_period=5,  # Save checkpoint every 5 epochs
        verbose=True,
    )

train_time = time.time() - t0
print(f"\nTRAINING COMPLETE ({train_time/60:.1f} min)")

# Save final weights to Drive immediately
for w in ["best.pt", "last.pt"]:
    src = Path(f"results/turbdetr_augmented/weights/{w}")
    if src.exists():
        shutil.copy2(src, DRIVE_CKPT / f"turbdetr_augmented_{w}")
        print(f"  {w} -> Drive")

# Save any epoch checkpoints
for ckpt in Path("results/turbdetr_augmented/weights").glob("epoch*.pt"):
    shutil.copy2(ckpt, DRIVE_CKPT / ckpt.name)
    print(f"  {ckpt.name} -> Drive")

print(f"\nCheckpoints saved to: {DRIVE_CKPT}")


Training from scratch...


100%|██████████| 63.4M/63.4M [00:00<00:00, 96.4MB/s]


New https://pypi.org/project/ultralytics/8.4.38 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.40 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=rtdetr-l.pt, data=configs/datasets/trash_icra19_augmented.yaml, epochs=20, time=None, patience=10, batch=16, imgsz=640, save=True, save_period=5, cache=False, device=0, workers=2, project=results, name=turbdetr_augmented, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embe

100%|██████████| 755k/755k [00:00<00:00, 6.60MB/s]


Overriding model.yaml nc=80 with nc=2
WARNING ⚠️ no model scale passed. Assuming scale='l'.

                   from  n    params  module                                       arguments                     
  0                  -1  1     25248  ultralytics.nn.modules.block.HGStem          [3, 32, 48]                   
  1                  -1  6    155072  ultralytics.nn.modules.block.HGBlock         [48, 48, 128, 3, 6]           
  2                  -1  1      1408  ultralytics.nn.modules.conv.DWConv           [128, 128, 3, 2, 1, False]    
  3                  -1  6    839296  ultralytics.nn.modules.block.HGBlock         [128, 96, 512, 3, 6]          
  4                  -1  1      5632  ultralytics.nn.modules.conv.DWConv           [512, 512, 3, 2, 1, False]    
  5                  -1  6   1695360  ultralytics.nn.modules.block.HGBlock         [512, 192, 1024, 5, 6, True, False]
  6                  -1  6   2055808  ultralytics.nn.modules.block.HGBlock         [1024, 192, 1024, 5, 

...


      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


  0%|          | 0/1431 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
       4/20      13.4G     0.3815     0.6397     0.2801         20        640: 100%|██████████| 1431/1431 [30:50<00:00,  1.29s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 26/26 [00:19<00:00,  1.35it/s]

                   all        820        923      0.432      0.423      0.411      0.271



      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


  0%|          | 0/1431 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
       5/20      13.5G     0.3611     0.5998     0.2632         29        640:  44%|████▍     | 632/1431 [13:36<16:50,  1.26s/it]

In [ ]:
# ============================================================
# CELL 7: Verify training completed + save training curves
# ============================================================
import shutil
from pathlib import Path

DRIVE_CKPT = Path("/content/drive/MyDrive/turb_detr_results/week2_checkpoints")

# Check if best.pt exists
best = Path("results/turbdetr_augmented/weights/best.pt")
if best.exists():
    print(f"best.pt exists ({best.stat().st_size / 1e6:.1f} MB)")
else:
    print("WARNING: best.pt not found!")
    # Try to use last.pt
    last = Path("results/turbdetr_augmented/weights/last.pt")
    if last.exists():
        print(f"  Using last.pt instead ({last.stat().st_size / 1e6:.1f} MB)")

# Save training curves
curves = Path("results/turbdetr_augmented/results.csv")
if curves.exists():
    shutil.copy2(curves, DRIVE_CKPT / "training_curves.csv")
    print("Training curves saved to Drive")

    # Print final metrics
    import csv
    with open(curves) as f:
        rows = list(csv.DictReader(f))
    if rows:
        last_row = rows[-1]
        print(f"\nFinal epoch metrics:")
        for k, v in last_row.items():
            k = k.strip()
            if any(x in k for x in ["map", "mAP", "precision", "recall"]):
                print(f"  {k}: {v}")


## Evaluation

**If you're running this in a SECOND session** (after resume), make sure Cells 1-4 have run to rebuild data and configs.


In [ ]:
# ============================================================
# CELL 8: Evaluate Turb-DETR on all conditions
# ============================================================
from pathlib import Path
from ultralytics import RTDETR

# Find best weights
BEST = None
for p in [
    Path("results/turbdetr_augmented/weights/best.pt"),
    Path("results/turbdetr_augmented/weights/last.pt"),
    Path("/content/drive/MyDrive/turb_detr_results/week2_checkpoints/turbdetr_augmented_best.pt"),
    Path("/content/drive/MyDrive/turb_detr_results/week2_checkpoints/turbdetr_augmented_last.pt"),
]:
    if p.exists():
        BEST = str(p)
        break

if BEST is None:
    raise FileNotFoundError("No trained weights found! Run training first.")
print(f"Using weights: {BEST}")

model = RTDETR(BEST)

# Track A: Clean
print("\n=== Clean test ===")
cr = model.val(data="configs/datasets/trash_icra19_clean.yaml", split="test", device=0)
td_clean = {"map50": cr.box.map50, "map50_95": cr.box.map, "prec": cr.box.mp, "rec": cr.box.mr, "ap": list(cr.box.ap50)}
print(f"  mAP@0.5: {td_clean['map50']:.4f}")

# Track B: Turbid
td_turbid = {}
for lv in ["light", "medium", "heavy"]:
    print(f"\n=== {lv} turbid ===")
    try:
        r = model.val(data=f"configs/datasets/trash_icra19_turbid_{lv}.yaml", split="test", device=0)
        td_turbid[lv] = {"map50": r.box.map50, "map50_95": r.box.map, "prec": r.box.mp, "rec": r.box.mr, "ap": list(r.box.ap50)}
        print(f"  mAP@0.5: {td_turbid[lv]['map50']:.4f}")
    except Exception as e:
        print(f"  ERROR: {e}")
        td_turbid[lv] = {"map50": 0, "map50_95": 0, "prec": 0, "rec": 0, "ap": [0, 0]}

print("\nAll evaluations complete")


In [ ]:
# ============================================================
# CELL 9: COMPARISON TABLE
# ============================================================
import csv, os
os.makedirs("results/tables", exist_ok=True)

# Week 1 baseline (from your run)
BL = {"clean": 0.9853, "light": 0.9735, "medium": 0.7368, "heavy": 0.1494}
BL_AP = {"heavy_plastic": 0.2155, "heavy_bio": 0.0834}

print()
print("="*80)
print("  VANILLA RT-DETR vs TURB-DETR (SimAM-injected)")
print("="*80)
print(f"  {'Condition':<18} {'Baseline':>12} {'Turb-DETR':>12} {'Delta':>10} {'Improve%':>10}")
print("-"*80)

rows = []
for label, key, val in [("Clean","clean",td_clean), ("Light (c=0.05)","light",td_turbid.get("light",{})),
                          ("Medium (c=0.15)","medium",td_turbid.get("medium",{})),
                          ("Heavy (c=0.30)","heavy",td_turbid.get("heavy",{}))]:
    bl = BL[key]
    td = val.get("map50", 0)
    d = td - bl
    pct = (d / bl * 100) if bl > 0 else 0
    ds = f"+{d:.4f}" if d >= 0 else f"{d:.4f}"
    ps = f"+{pct:.1f}%" if pct >= 0 else f"{pct:.1f}%"
    print(f"  {label:<18} {bl:>12.4f} {td:>12.4f} {ds:>10} {ps:>10}")
    rows.append({"condition": key, "baseline": bl, "turbdetr": td, "delta": d})

print("="*80)

# Per-class at heavy
print(f"\n  Per-class AP@0.5 at Heavy Turbidity:")
td_h_ap = td_turbid.get("heavy", {}).get("ap", [0, 0])
for i, (nm, bl_ap) in enumerate([("plastic", 0.2155), ("bio", 0.0834)]):
    td_ap = td_h_ap[i] if i < len(td_h_ap) else 0
    d = td_ap - bl_ap
    print(f"    {nm}: {bl_ap:.4f} -> {td_ap:.4f} ({d:+.4f})")

print("="*80)

# Verdict
hi = td_turbid.get("heavy", {}).get("map50", 0) - BL["heavy"]
print()
if hi > 0.15:
    print(f"  EXCELLENT (+{hi:.4f} mAP at heavy). Strong paper result.")
elif hi > 0.05:
    print(f"  GOOD (+{hi:.4f} mAP at heavy). Publishable with ablation.")
elif hi > 0.02:
    print(f"  MODERATE (+{hi:.4f} mAP). Need to isolate SimAM vs augmentation effect.")
else:
    print(f"  WEAK (+{hi:.4f} mAP). Try different injection point or lambda.")
print("="*80)

# Save
csv_path = "results/tables/baseline_vs_turbdetr.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["condition", "baseline", "turbdetr", "delta"])
    w.writeheader(); w.writerows(rows)
print(f"\n  Saved: {csv_path}")
print("\n  *** SEND THIS TABLE TO CLAUDE ***")


In [ ]:
# ============================================================
# CELL 10: Save everything to Drive
# ============================================================
import shutil
from pathlib import Path

SAVE = Path("/content/drive/MyDrive/turb_detr_results/week2")
SAVE.mkdir(parents=True, exist_ok=True)

# Weights
for w in ["best.pt", "last.pt"]:
    src = Path(f"results/turbdetr_augmented/weights/{w}")
    if src.exists():
        shutil.copy2(src, SAVE / f"turbdetr_augmented_{w}")
        print(f"  {w} saved")

# Tables
for f in Path("results/tables").glob("*.csv"):
    shutil.copy2(f, SAVE / f.name)
    print(f"  {f.name} saved")

# Training curves
src = Path("results/turbdetr_augmented/results.csv")
if src.exists():
    shutil.copy2(src, SAVE / "turbdetr_training_curves.csv")
    print("  training curves saved")

print(f"\nAll saved to: {SAVE}")
print("Session can disconnect safely.")
